# Week 6 Notebook: Physics-informed MLP for Three-phase Microgrid N-1 Screening

本 notebook 从 Week 5 的传统 ML baseline 出发，构建一个 **limit-aware / physics-informed MLP**。这里的 physics-informed 不是直接把三相潮流方程嵌入神经网络求解，而是把三相静态安全分析中的约束结构加入训练目标：

- binary violation classification；
- severity regression；
- voltage / loading / VUF / service-loss component margins；
- severity-component consistency；
- probability-severity consistency；
- pairwise ranking loss；
- high-recall threshold selection；
- random holdout + scenario-group CV + contingency-group CV。

> 重要原则：**post-contingency 结果可以作为训练 target，但绝不能作为部署时输入 feature。**

## 0. Imports and configuration

本节固定随机种子、设置输出目录，并加载 Week 5 生成的 `week5_ml_input_dataset.csv`。

In [ ]:
from __future__ import annotations

import json
import math
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 20260517
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)

DATA_PATH = Path('/mnt/data/week5_outputs_complete/week5_ml_input_dataset.csv')
OUT_DIR = Path('/mnt/data/week6_outputs_complete')
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RECALL = 0.95
V_LOW = 0.95
V_HIGH = 1.05
LOADING_MAX_PERCENT = 100.0
VUF_MAX_PERCENT = 2.0

print('Data path:', DATA_PATH)
print('Output directory:', OUT_DIR)
print('PyTorch version:', torch.__version__)

## 1. Load Week 5 dataset

Week 5 的数据已经把 Week 4 的 N-1 label 与 Week 3 的 base-case features 合并。我们继续使用同一份数据，以保证 Week 5 baseline 和 Week 6 PI-MLP 可以公平比较。

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Cannot find {DATA_PATH}. Run Week 5 notebook first or update DATA_PATH.'
    )

df = pd.read_csv(DATA_PATH)
if 'row_id' not in df.columns:
    df.insert(0, 'row_id', [f'r{i:04d}' for i in range(len(df))])

print('Dataset shape:', df.shape)
print('Unsafe / safe counts:')
print(df['violation_label'].astype(int).value_counts().rename(index={0:'safe', 1:'unsafe'}))

df.head()

## 2. Target construction: from violation labels to physics-informed auxiliary targets

Week 6 的关键是把一个二元 label 扩展成多个具有物理含义的监督目标。二元标签只回答“是否 unsafe”，而 physics-informed auxiliary targets 进一步告诉模型风险来自哪个约束。

对已收敛的 N-1 样本，我们从 post-contingency 指标构造 component margin：

$$
m_{V,low}=\left[\frac{0.95-V_{min}}{0.95}\right]_+
$$

$$
m_{V,high}=\left[\frac{V_{max}-1.05}{1.05}\right]_+
$$

$$
m_I=\left[\frac{L_{max}}{100}-1\right]_+
$$

$$
m_U=\left[\frac{U_{max}}{2}-1\right]_+
$$

对于 service-loss / no-slack 样本，连续潮流指标可能是 NaN；这类风险由 service margin 表示。最终构造：

$$
s^{PI}=\max\{m_{V,low},m_{V,high},m_I,m_U,m_{service},m_{critical},m_{noslack},s_{raw}\}.
$$

教学重点：post-contingency 结果可以作为训练 target，但不能作为部署时输入 feature。

In [ ]:
# Ensure boolean labels are 0/1 integers.
flag_cols = [
    'voltage_low_violation',
    'voltage_high_violation',
    'loading_violation',
    'vuf_violation',
    'non_convergence_violation',
    'service_loss_violation',
    'critical_service_loss_violation',
    'violation_label',
]
for col in flag_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required target column: {col}')
    df[col] = df[col].astype(int)

min_vm = df['min_vm_pu']
max_vm = df['max_vm_pu']
max_loading = df['max_line_loading_percent']
max_vuf = df['max_vuf_percent']

# Component margins. NaN post-contingency metrics are not imputed as fake electrical values.
df['target_vlow_margin'] = np.where(
    min_vm.notna(),
    np.maximum((V_LOW - min_vm) / V_LOW, 0.0),
    df['voltage_low_violation'].astype(float),
)
df['target_vhigh_margin'] = np.where(
    max_vm.notna(),
    np.maximum((max_vm - V_HIGH) / V_HIGH, 0.0),
    df['voltage_high_violation'].astype(float),
)
df['target_loading_margin'] = np.where(
    max_loading.notna(),
    np.maximum(max_loading / LOADING_MAX_PERCENT - 1.0, 0.0),
    df['loading_violation'].astype(float),
)
df['target_vuf_margin'] = np.where(
    max_vuf.notna(),
    np.maximum(max_vuf / VUF_MAX_PERCENT - 1.0, 0.0),
    df['vuf_violation'].astype(float),
)
df['target_service_margin'] = df[
    ['non_convergence_violation', 'service_loss_violation', 'critical_service_loss_violation']
].max(axis=1).astype(float)

component_cols = [
    'target_vlow_margin',
    'target_vhigh_margin',
    'target_loading_margin',
    'target_vuf_margin',
    'target_service_margin',
]

df['target_component_max_margin'] = df[component_cols].max(axis=1)
df['target_pi_severity'] = np.maximum(
    df['severity_score'].fillna(0).astype(float),
    df['target_component_max_margin'].astype(float),
)

# Proof: PI severity should dominate every component target.
assert (df['target_pi_severity'] >= df['target_component_max_margin'] - 1e-12).all()

# Proof: label and raw severity should be consistent for the generated Week 4 dataset.
assert ((df['violation_label'] == 0) == (df['severity_score'].fillna(0) == 0)).all()
assert ((df['violation_label'] == 1) == (df['severity_score'].fillna(0) > 0)).all()

component_summary = df[component_cols + ['target_pi_severity']].describe().T
component_summary.to_csv(OUT_DIR / 'week6_component_target_summary.csv')
component_summary

## 3. No-leakage feature design

Post-contingency quantities are allowed as **targets** but forbidden as **input features**. This is the most important proof in Week 6.

In [ ]:
identifier_cols = ['row_id', 'scenario_id', 'contingency_id']

leakage_cols = [
    # Post-contingency solver status and physical outcomes
    'converged',
    'min_vm_pu',
    'max_vm_pu',
    'max_vuf_percent',
    'max_line_loading_percent',
    'unserved_load_mw',
    'critical_unserved_load_mw',
    # Violation labels and component flags
    'voltage_low_violation',
    'voltage_high_violation',
    'loading_violation',
    'vuf_violation',
    'non_convergence_violation',
    'service_loss_violation',
    'critical_service_loss_violation',
    'violation_label',
    'severity_score',
    # Week 6 target columns
    'target_pi_severity',
    'target_component_max_margin',
    *component_cols,
]

feature_cols = [c for c in df.columns if c not in identifier_cols + leakage_cols]

# Strong no-leakage proof based on exact column membership.
assert not set(feature_cols).intersection(leakage_cols)

# Stronger semantic check: suspicious substrings should not appear unless they are explicitly base-case features.
suspicious_tokens = ['min_vm', 'max_vuf', 'violation', 'severity', 'unserved', 'converged']
violating_feature_names = []
for c in feature_cols:
    for token in suspicious_tokens:
        if token in c and not c.startswith('base_'):
            violating_feature_names.append(c)
assert len(violating_feature_names) == 0, violating_feature_names

categorical_cols = [c for c in feature_cols if df[c].dtype == 'object']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

feature_list = pd.DataFrame({
    'feature': feature_cols,
    'kind': ['categorical' if c in categorical_cols else 'numeric' for c in feature_cols],
})
feature_list.to_csv(OUT_DIR / 'week6_feature_list.csv', index=False)

print('Number of input features before one-hot:', len(feature_cols))
print('Categorical columns:', categorical_cols)
print('Numeric columns:', len(numeric_cols))
feature_list.head(20)

## 4. Train / validation / test split

阈值必须只在 validation set 上选择，test set 只用于最终报告。

In [ ]:
y_all = df['violation_label'].to_numpy(dtype=int)
indices = np.arange(len(df))

trainval_idx, test_idx = train_test_split(
    indices,
    test_size=0.25,
    stratify=y_all,
    random_state=SEED,
)
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.30,
    stratify=y_all[trainval_idx],
    random_state=SEED + 1,
)

assert set(train_idx).isdisjoint(val_idx)
assert set(train_idx).isdisjoint(test_idx)
assert set(val_idx).isdisjoint(test_idx)
assert sorted(np.r_[train_idx, val_idx, test_idx].tolist()) == list(range(len(df)))

split_summary = pd.DataFrame([
    {'split': 'train', 'n': len(train_idx), 'unsafe': int(y_all[train_idx].sum()), 'safe': int((1-y_all[train_idx]).sum())},
    {'split': 'validation', 'n': len(val_idx), 'unsafe': int(y_all[val_idx].sum()), 'safe': int((1-y_all[val_idx]).sum())},
    {'split': 'test', 'n': len(test_idx), 'unsafe': int(y_all[test_idx].sum()), 'safe': int((1-y_all[test_idx]).sum())},
])
split_summary.to_csv(OUT_DIR / 'week6_split_summary.csv', index=False)
split_summary

## 5. Preprocessing pipeline

只在 training set 上 fit scaler / one-hot encoder，再 transform validation 和 test。这样可以避免 preprocessing leakage。

In [ ]:
def make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical_cols),
    ], remainder='drop')

preprocessor = make_preprocessor()
X_train = preprocessor.fit_transform(df.loc[train_idx, feature_cols])
X_val = preprocessor.transform(df.loc[val_idx, feature_cols])
X_test = preprocessor.transform(df.loc[test_idx, feature_cols])

assert np.isfinite(X_train).all()
assert np.isfinite(X_val).all()
assert np.isfinite(X_test).all()

print('Encoded feature dimension:', X_train.shape[1])
print('Train / val / test:', X_train.shape, X_val.shape, X_test.shape)

## 6. Tensor packing utilities

为了同时训练 classification、severity 和 component heads，我们把每个 split 打包成四类 tensor：

```text
X      input features
y      binary label
sev    PI severity target
comp   component margin targets
```

In [ ]:
def pack_tensors(idx: np.ndarray, X_encoded: np.ndarray) -> Dict[str, torch.Tensor]:
    return {
        'X': torch.tensor(X_encoded, dtype=torch.float32),
        'y': torch.tensor(df.loc[idx, 'violation_label'].to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1),
        'sev': torch.tensor(df.loc[idx, 'target_pi_severity'].to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1),
        'comp': torch.tensor(df.loc[idx, component_cols].to_numpy(dtype=np.float32), dtype=torch.float32),
        'idx': torch.tensor(idx, dtype=torch.long),
    }

train_data = pack_tensors(train_idx, X_train)
val_data = pack_tensors(val_idx, X_val)
test_data = pack_tensors(test_idx, X_test)

for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    assert data['X'].shape[0] == data['y'].shape[0] == data['sev'].shape[0] == data['comp'].shape[0]
    assert data['comp'].shape[1] == len(component_cols)
    assert torch.isfinite(data['X']).all()
    assert torch.isfinite(data['sev']).all()
    assert torch.isfinite(data['comp']).all()
    print(name, {k: tuple(v.shape) for k, v in data.items() if k != 'idx'})

## 7. Model definitions

我们比较两个模型：

1. `BaselineMLP`: Week 5 风格，只做 BCE classification；
2. `PIMLP`: shared encoder + classification head + severity head + component margin head。

In [ ]:
class BaselineMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: Tuple[int, ...] = (32, 16), dropout: float = 0.05):
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.cls = nn.Linear(prev, 1)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.encoder(x)
        return {'logit': self.cls(z)}


class PIMLP(nn.Module):
    def __init__(self, input_dim: int, hidden: Tuple[int, ...] = (32, 16), n_comp: int = 5, dropout: float = 0.05):
        super().__init__()
        layers: List[nn.Module] = []
        prev = input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.cls = nn.Linear(prev, 1)
        self.sev = nn.Linear(prev, 1)
        self.comp = nn.Linear(prev, n_comp)

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        z = self.encoder(x)
        return {
            'logit': self.cls(z),
            'sev': torch.nn.functional.softplus(self.sev(z)),
            'comp': torch.nn.functional.softplus(self.comp(z)),
        }


def model_output_shape_proof() -> pd.DataFrame:
    x_small = train_data['X'][:4]
    d = train_data['X'].shape[1]
    base = BaselineMLP(d)
    pi = PIMLP(d, n_comp=len(component_cols))
    out_base = base(x_small)
    out_pi = pi(x_small)
    checks = [
        {'check': 'baseline_logit_shape', 'passed': tuple(out_base['logit'].shape) == (4, 1), 'detail': str(tuple(out_base['logit'].shape))},
        {'check': 'pi_logit_shape', 'passed': tuple(out_pi['logit'].shape) == (4, 1), 'detail': str(tuple(out_pi['logit'].shape))},
        {'check': 'pi_severity_shape', 'passed': tuple(out_pi['sev'].shape) == (4, 1), 'detail': str(tuple(out_pi['sev'].shape))},
        {'check': 'pi_component_shape', 'passed': tuple(out_pi['comp'].shape) == (4, len(component_cols)), 'detail': str(tuple(out_pi['comp'].shape))},
        {'check': 'pi_nonnegative_aux_outputs', 'passed': bool((out_pi['sev'] >= 0).all() and (out_pi['comp'] >= 0).all()), 'detail': 'softplus heads'},
    ]
    return pd.DataFrame(checks)

shape_proof = model_output_shape_proof()
assert shape_proof['passed'].all()
shape_proof

## 8. Physics-informed loss functions

PI-MLP 的总损失为：

$$
\mathcal{L}
=
\mathcal{L}_{cls}
+
\lambda_s\mathcal{L}_{sev}
+
\lambda_c\mathcal{L}_{comp}
+
\lambda_{cons}\mathcal{L}_{cons}
+
\lambda_p\mathcal{L}_{prob-sev}
+
\lambda_r\mathcal{L}_{rank}
$$

其中 consistency loss 要求：

$$
\hat s \ge \max_k \hat m_k
$$

In [ ]:
def make_loader(data: Dict[str, torch.Tensor], batch_size: int = 32, shuffle: bool = True, seed: int = SEED) -> DataLoader:
    ds = TensorDataset(data['X'], data['y'], data['sev'], data['comp'])
    gen = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=gen)


def pairwise_ranking_loss(scores: torch.Tensor, target_sev: torch.Tensor, margin: float = 0.05) -> torch.Tensor:
    s = scores.view(-1, 1)
    t = target_sev.view(-1, 1)
    target_diff = t - t.T
    mask = target_diff > 1e-6
    if mask.sum() == 0:
        return scores.sum() * 0.0
    pred_diff = s - s.T
    return torch.relu(margin - pred_diff[mask]).mean()


DEFAULT_PI_WEIGHTS = {
    'sev': 0.50,
    'comp': 0.25,
    'cons': 0.20,
    'prob_sev': 0.10,
    'rank': 0.20,
}


def compute_loss(
    model_kind: str,
    out: Dict[str, torch.Tensor],
    yb: torch.Tensor,
    sevb: torch.Tensor,
    compb: torch.Tensor,
    pos_weight: torch.Tensor,
    pi_weights: Optional[Dict[str, float]] = None,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    cls_loss = bce(out['logit'], yb)
    parts = {'cls': float(cls_loss.detach().cpu())}
    if model_kind == 'baseline':
        return cls_loss, parts

    if pi_weights is None:
        pi_weights = DEFAULT_PI_WEIGHTS

    sev_loss = nn.functional.smooth_l1_loss(torch.log1p(out['sev']), torch.log1p(sevb))
    comp_loss = nn.functional.smooth_l1_loss(torch.log1p(out['comp']), torch.log1p(compb))
    cons_loss = torch.relu(out['comp'].max(dim=1, keepdim=True).values - out['sev']).pow(2).mean()
    prob_cons_loss = nn.functional.mse_loss(
        torch.sigmoid(out['logit']),
        torch.sigmoid(8.0 * (out['sev'] - 0.05)),
    )
    rank_loss = pairwise_ranking_loss(out['sev'], sevb)

    total = (
        cls_loss
        + pi_weights.get('sev', 0.0) * sev_loss
        + pi_weights.get('comp', 0.0) * comp_loss
        + pi_weights.get('cons', 0.0) * cons_loss
        + pi_weights.get('prob_sev', 0.0) * prob_cons_loss
        + pi_weights.get('rank', 0.0) * rank_loss
    )
    parts.update({
        'sev': float(sev_loss.detach().cpu()),
        'comp': float(comp_loss.detach().cpu()),
        'cons': float(cons_loss.detach().cpu()),
        'prob_sev': float(prob_cons_loss.detach().cpu()),
        'rank': float(rank_loss.detach().cpu()),
    })
    return total, parts


def finite_gradient_proof() -> pd.DataFrame:
    d = train_data['X'].shape[1]
    model = PIMLP(d, n_comp=len(component_cols))
    pos = float(train_data['y'].sum().item())
    neg = float(len(train_data['y']) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32)
    loader = make_loader(train_data, batch_size=16, shuffle=False)
    xb, yb, sevb, compb = next(iter(loader))
    out = model(xb)
    loss, parts = compute_loss('pi', out, yb, sevb, compb, pos_weight, DEFAULT_PI_WEIGHTS)
    loss.backward()
    finite_grads = True
    for p in model.parameters():
        if p.grad is not None and not torch.isfinite(p.grad).all():
            finite_grads = False
    checks = [
        {'check': 'finite_loss', 'passed': bool(torch.isfinite(loss)), 'detail': f'{float(loss):.6f}'},
        {'check': 'finite_gradients', 'passed': finite_grads, 'detail': json.dumps(parts)},
    ]
    return pd.DataFrame(checks)

grad_proof = finite_gradient_proof()
assert grad_proof['passed'].all()
grad_proof

## 9. Training and evaluation utilities

本节定义训练函数、阈值选择函数和手算混淆矩阵函数。

In [ ]:
def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_torch_model(
    model_kind: str,
    train_tensors: Dict[str, torch.Tensor],
    val_tensors: Dict[str, torch.Tensor],
    input_dim: int,
    pi_weights: Optional[Dict[str, float]] = None,
    max_epochs: int = 200,
    lr: float = 0.01,
    seed: int = SEED,
    patience: int = 30,
) -> Tuple[nn.Module, pd.DataFrame]:
    set_all_seeds(seed)
    if model_kind == 'baseline':
        model = BaselineMLP(input_dim)
    elif model_kind == 'pi':
        model = PIMLP(input_dim, n_comp=len(component_cols))
    else:
        raise ValueError(model_kind)

    pos = float(train_tensors['y'].sum().item())
    neg = float(len(train_tensors['y']) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    train_loader = make_loader(train_tensors, batch_size=32, shuffle=True, seed=seed)
    val_loader = make_loader(val_tensors, batch_size=128, shuffle=False, seed=seed)

    records = []
    best_state = None
    best_val = float('inf')
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_total = 0.0
        train_n = 0
        for xb, yb, sevb, compb in train_loader:
            out = model(xb)
            loss, parts = compute_loss(model_kind, out, yb, sevb, compb, pos_weight, pi_weights)
            if not torch.isfinite(loss):
                raise RuntimeError('Non-finite training loss detected')
            optimizer.zero_grad()
            loss.backward()
            for p in model.parameters():
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    raise RuntimeError('Non-finite gradient detected')
            optimizer.step()
            train_total += float(loss.item()) * len(xb)
            train_n += len(xb)

        model.eval()
        val_total = 0.0
        val_n = 0
        with torch.no_grad():
            for xb, yb, sevb, compb in val_loader:
                out = model(xb)
                val_loss, val_parts = compute_loss(model_kind, out, yb, sevb, compb, pos_weight, pi_weights)
                val_total += float(val_loss.item()) * len(xb)
                val_n += len(xb)

        train_loss = train_total / train_n
        val_loss = val_total / val_n
        records.append({'epoch': epoch, 'model_kind': model_kind, 'train_loss': train_loss, 'val_loss': val_loss})

        if val_loss < best_val - 1e-7:
            best_val = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
        if bad_epochs >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, pd.DataFrame(records)


def predict_model(model: nn.Module, X_encoded: np.ndarray) -> Dict[str, np.ndarray]:
    model.eval()
    with torch.no_grad():
        out = model(torch.tensor(X_encoded, dtype=torch.float32))
        result = {'prob': torch.sigmoid(out['logit']).cpu().numpy().ravel()}
        if 'sev' in out:
            result['sev_pred'] = out['sev'].cpu().numpy().ravel()
            result['comp_pred'] = out['comp'].cpu().numpy()
        return result


def metrics_at_threshold(y_true: np.ndarray, score: np.ndarray, threshold: float) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score, dtype=float)
    pred = (score >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    recall = tp / (tp + fn) if (tp + fn) else np.nan
    precision = tp / (tp + fp) if (tp + fp) else np.nan
    fnr = fn / (tp + fn) if (tp + fn) else np.nan
    return {
        'threshold': float(threshold),
        'TP': tp,
        'FP': fp,
        'TN': tn,
        'FN': fn,
        'recall': recall,
        'precision': precision,
        'fnr': fnr,
        'pf_calls_saved_fraction': (tn + fn) / len(y_true),
        'missed_violations': fn,
    }


def add_auc_metrics(metrics: Dict[str, float], y_true: np.ndarray, score: np.ndarray) -> Dict[str, float]:
    out = dict(metrics)
    if len(np.unique(y_true)) > 1:
        out['auprc'] = float(average_precision_score(y_true, score))
        out['auroc'] = float(roc_auc_score(y_true, score))
    else:
        out['auprc'] = np.nan
        out['auroc'] = np.nan
    return out


def select_threshold_for_high_recall(y_true: np.ndarray, score: np.ndarray, target_recall: float = TARGET_RECALL) -> Tuple[float, pd.DataFrame]:
    thresholds = np.unique(np.r_[0.0, np.linspace(0.0, 1.0, 201), score])
    rows = [metrics_at_threshold(y_true, score, t) for t in thresholds]
    table = pd.DataFrame(rows)
    eligible = table[table['recall'] >= target_recall]
    if eligible.empty:
        chosen = table.sort_values(['recall', 'pf_calls_saved_fraction'], ascending=[False, False]).iloc[0]
    else:
        chosen = eligible.sort_values(['pf_calls_saved_fraction', 'precision'], ascending=[False, False]).iloc[0]
    return float(chosen['threshold']), table

## 10. Train BaselineMLP and PI-MLP

为了保持教学速度，hidden layer 很小。真实论文实验可以扩展 hidden size、随机种子数量和超参数搜索。

In [ ]:
input_dim = X_train.shape[1]

baseline_model, hist_baseline = train_torch_model(
    'baseline', train_data, val_data, input_dim=input_dim, max_epochs=160, lr=0.01, seed=SEED, patience=30
)

pi_full_weights = DEFAULT_PI_WEIGHTS.copy()
pi_model, hist_pi = train_torch_model(
    'pi', train_data, val_data, input_dim=input_dim, pi_weights=pi_full_weights, max_epochs=220, lr=0.01, seed=SEED + 2, patience=35
)

hist_baseline['model'] = 'BaselineMLP'
hist_pi['model'] = 'PI-MLP full'
training_curves = pd.concat([hist_baseline, hist_pi], ignore_index=True)
training_curves.to_csv(OUT_DIR / 'week6_training_curves.csv', index=False)

training_curves.groupby('model').tail(1)

## 11. High-recall threshold selection and test evaluation

阈值选择只使用 validation set；test set 不参与调参。

In [ ]:
y_val = y_all[val_idx]
y_test = y_all[test_idx]

val_baseline = predict_model(baseline_model, X_val)['prob']
test_baseline = predict_model(baseline_model, X_test)['prob']

val_pi = predict_model(pi_model, X_val)['prob']
test_pi_outputs = predict_model(pi_model, X_test)
test_pi = test_pi_outputs['prob']

holdout_rows = []
tradeoff_tables = []

for model_name, val_score, test_score in [
    ('BaselineMLP', val_baseline, test_baseline),
    ('PI-MLP full', val_pi, test_pi),
]:
    threshold, tradeoff = select_threshold_for_high_recall(y_val, val_score, TARGET_RECALL)
    tradeoff['model'] = model_name
    tradeoff_tables.append(tradeoff)
    metrics = metrics_at_threshold(y_test, test_score, threshold)
    metrics = add_auc_metrics(metrics, y_test, test_score)
    metrics.update({'model': model_name, 'selected_threshold': threshold, 'threshold_source': 'validation'})
    holdout_rows.append(metrics)

holdout_results = pd.DataFrame(holdout_rows)
screening_tradeoff = pd.concat(tradeoff_tables, ignore_index=True)

holdout_results.to_csv(OUT_DIR / 'week6_holdout_results.csv', index=False)
screening_tradeoff.to_csv(OUT_DIR / 'week6_screening_tradeoff.csv', index=False)

holdout_results

## 12. Manual confusion-matrix proof

这里再次手算 confusion matrix，避免学生把 recall / FNR / saved PF calls 当成黑箱指标。

In [ ]:
pi_threshold = float(holdout_results.loc[holdout_results['model'] == 'PI-MLP full', 'selected_threshold'].iloc[0])
manual_metrics = metrics_at_threshold(y_test, test_pi, pi_threshold)

assert manual_metrics['TP'] + manual_metrics['FP'] + manual_metrics['TN'] + manual_metrics['FN'] == len(test_idx)
assert abs(manual_metrics['recall'] + manual_metrics['fnr'] - 1.0) < 1e-12

manual_confusion = pd.DataFrame([manual_metrics])
manual_confusion.to_csv(OUT_DIR / 'week6_manual_confusion_matrix_proof.csv', index=False)
manual_confusion

## 13. Loss ablation: with and without ranking

本节展示最小 ablation。完整论文版可以进一步比较：without component、without consistency、different \(\lambda\) settings。

In [ ]:
pi_no_rank_weights = DEFAULT_PI_WEIGHTS.copy()
pi_no_rank_weights['rank'] = 0.0

pi_no_rank_model, hist_pi_no_rank = train_torch_model(
    'pi', train_data, val_data, input_dim=input_dim, pi_weights=pi_no_rank_weights, max_epochs=180, lr=0.01, seed=SEED + 3, patience=30
)

ablation_models = [
    ('BaselineMLP', baseline_model, None),
    ('PI-MLP no-rank', pi_no_rank_model, pi_no_rank_weights),
    ('PI-MLP full', pi_model, pi_full_weights),
]

ablation_rows = []
for name, model, _weights in ablation_models:
    val_score = predict_model(model, X_val)['prob']
    test_score = predict_model(model, X_test)['prob']
    threshold, _ = select_threshold_for_high_recall(y_val, val_score, TARGET_RECALL)
    metrics = metrics_at_threshold(y_test, test_score, threshold)
    metrics = add_auc_metrics(metrics, y_test, test_score)
    metrics.update({'model': name, 'selected_threshold': threshold})
    ablation_rows.append(metrics)

loss_ablation_summary = pd.DataFrame(ablation_rows)
loss_ablation_summary.to_csv(OUT_DIR / 'week6_loss_ablation_summary.csv', index=False)
loss_ablation_summary

## 14. Test predictions and auxiliary head inspection

PI-MLP 不仅输出概率，还输出 severity 和 component margin。下表用于检查模型解释性。

In [ ]:
test_predictions = df.loc[test_idx, [
    'row_id', 'scenario_id', 'contingency_id', 'contingency_type',
    'violation_label', 'severity_score', 'target_pi_severity', *component_cols,
]].copy()

test_predictions['baseline_prob'] = test_baseline
test_predictions['pi_prob'] = test_pi
test_predictions['pi_severity_pred'] = test_pi_outputs['sev_pred']
for j, col in enumerate(component_cols):
    test_predictions['pred_' + col.replace('target_', '')] = test_pi_outputs['comp_pred'][:, j]

test_predictions = test_predictions.sort_values('pi_prob', ascending=False)
test_predictions.to_csv(OUT_DIR / 'week6_test_predictions.csv', index=False)
test_predictions.head(10)

## 15. Scenario-group and contingency-group cross-validation

这一步是 Week 6 的 cross-validation proof：

- scenario-group CV：测试未见过 operating scenario；
- contingency-group CV：测试未见过 contingency。

为了课堂运行速度，默认每类 group CV 做 3 folds。

In [ ]:
def fit_transform_for_indices(tr_idx: np.ndarray, va_idx: np.ndarray, te_idx: np.ndarray):
    pp = make_preprocessor()
    X_tr = pp.fit_transform(df.loc[tr_idx, feature_cols])
    X_va = pp.transform(df.loc[va_idx, feature_cols])
    X_te = pp.transform(df.loc[te_idx, feature_cols])
    return X_tr, X_va, X_te


def make_local_tensors(idx: np.ndarray, X_encoded: np.ndarray) -> Dict[str, torch.Tensor]:
    return {
        'X': torch.tensor(X_encoded, dtype=torch.float32),
        'y': torch.tensor(df.loc[idx, 'violation_label'].to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1),
        'sev': torch.tensor(df.loc[idx, 'target_pi_severity'].to_numpy(dtype=np.float32), dtype=torch.float32).view(-1, 1),
        'comp': torch.tensor(df.loc[idx, component_cols].to_numpy(dtype=np.float32), dtype=torch.float32),
        'idx': torch.tensor(idx, dtype=torch.long),
    }


def group_cv_experiment(group_col: str, max_folds: int = 3, max_epochs: int = 80) -> pd.DataFrame:
    groups = df[group_col].to_numpy()
    n_splits = min(max_folds, len(np.unique(groups)))
    gkf = GroupKFold(n_splits=n_splits)
    rows = []
    for fold, (train_group_idx, test_group_idx) in enumerate(gkf.split(df[feature_cols], y_all, groups=groups), start=1):
        train_groups = set(groups[train_group_idx])
        test_groups = set(groups[test_group_idx])
        assert train_groups.isdisjoint(test_groups)

        # Internal validation split inside the training groups.
        try:
            tr_idx, va_idx = train_test_split(
                train_group_idx,
                test_size=0.25,
                stratify=y_all[train_group_idx],
                random_state=SEED + fold,
            )
        except ValueError:
            tr_idx, va_idx = train_test_split(train_group_idx, test_size=0.25, random_state=SEED + fold)

        X_tr, X_va, X_te = fit_transform_for_indices(tr_idx, va_idx, test_group_idx)
        tr_t = make_local_tensors(tr_idx, X_tr)
        va_t = make_local_tensors(va_idx, X_va)

        model, _hist = train_torch_model(
            'pi', tr_t, va_t, input_dim=X_tr.shape[1], pi_weights=pi_full_weights,
            max_epochs=max_epochs, lr=0.01, seed=SEED + 100 + fold, patience=20,
        )

        val_score = predict_model(model, X_va)['prob']
        test_score = predict_model(model, X_te)['prob']
        threshold, _ = select_threshold_for_high_recall(y_all[va_idx], val_score, TARGET_RECALL)
        metrics = metrics_at_threshold(y_all[test_group_idx], test_score, threshold)
        metrics = add_auc_metrics(metrics, y_all[test_group_idx], test_score)
        metrics.update({
            'cv_type': group_col,
            'fold': fold,
            'n_train': len(tr_idx),
            'n_val': len(va_idx),
            'n_test': len(test_group_idx),
            'n_test_groups': len(test_groups),
            'test_groups': '|'.join(map(str, sorted(test_groups))),
            'selected_threshold': threshold,
            'group_disjoint': True,
        })
        rows.append(metrics)
    return pd.DataFrame(rows)

scenario_cv = group_cv_experiment('scenario_id', max_folds=3, max_epochs=80)
contingency_cv = group_cv_experiment('contingency_id', max_folds=3, max_epochs=80)

group_cv_results = pd.concat([scenario_cv, contingency_cv], ignore_index=True)
group_cv_summary = group_cv_results.groupby('cv_type').agg(
    folds=('fold', 'count'),
    mean_recall=('recall', 'mean'),
    mean_fnr=('fnr', 'mean'),
    mean_pf_calls_saved=('pf_calls_saved_fraction', 'mean'),
    mean_auprc=('auprc', 'mean'),
).reset_index()

group_cv_results.to_csv(OUT_DIR / 'week6_group_cv_results.csv', index=False)
group_cv_summary.to_csv(OUT_DIR / 'week6_group_cv_summary.csv', index=False)

group_cv_summary

## 16. Reproducibility proof

用完全相同的随机种子重新训练 PI-MLP，检查 test probability 是否一致。

In [ ]:
pi_model_repeat, _hist_repeat = train_torch_model(
    'pi', train_data, val_data, input_dim=input_dim, pi_weights=pi_full_weights, max_epochs=220, lr=0.01, seed=SEED + 2, patience=35
)
repeat_prob = predict_model(pi_model_repeat, X_test)['prob']
repro_max_abs_diff = float(np.max(np.abs(repeat_prob - test_pi)))

reproducibility_proof = pd.DataFrame([{
    'check': 'fixed_seed_reproducibility',
    'max_abs_probability_diff': repro_max_abs_diff,
    'passed': repro_max_abs_diff < 1e-6,
}])
reproducibility_proof.to_csv(OUT_DIR / 'week6_reproducibility_proof.csv', index=False)
assert reproducibility_proof['passed'].all()
reproducibility_proof

## 17. Plots

生成 Week 6 的主要教学图表。

In [ ]:
# 1) Training curves
plt.figure(figsize=(7, 4))
for model_name, g in training_curves.groupby('model'):
    plt.plot(g['epoch'], g['val_loss'], label=f'{model_name} val')
plt.xlabel('Epoch')
plt.ylabel('Validation loss')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'week6_training_curves.png', dpi=160)

# 2) Precision-recall curves on test set
plt.figure(figsize=(7, 4))
for model_name, score in [('BaselineMLP', test_baseline), ('PI-MLP full', test_pi)]:
    precision, recall, _ = precision_recall_curve(y_test, score)
    plt.plot(recall, precision, label=model_name)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'week6_precision_recall_curves.png', dpi=160)

# 3) Screening tradeoff for PI model on validation set
pi_tradeoff = screening_tradeoff[screening_tradeoff['model'] == 'PI-MLP full'].sort_values('pf_calls_saved_fraction')
plt.figure(figsize=(7, 4))
plt.plot(pi_tradeoff['pf_calls_saved_fraction'], pi_tradeoff['missed_violations'])
plt.xlabel('PF calls saved fraction on validation set')
plt.ylabel('Missed violations')
plt.tight_layout()
plt.savefig(OUT_DIR / 'week6_saved_calls_vs_missed_violations.png', dpi=160)

# 4) Severity scatter
plt.figure(figsize=(6, 4))
plt.scatter(test_predictions['target_pi_severity'], test_predictions['pi_severity_pred'])
plt.xlabel('True PI severity target')
plt.ylabel('Predicted severity')
plt.tight_layout()
plt.savefig(OUT_DIR / 'week6_true_vs_predicted_severity.png', dpi=160)

# 5) Component target distribution
plt.figure(figsize=(7, 4))
df[component_cols].sum().sort_values().plot(kind='barh')
plt.xlabel('Sum of component margins over all samples')
plt.tight_layout()
plt.savefig(OUT_DIR / 'week6_component_target_distribution.png', dpi=160)

## 18. Validation summary

本节汇总 Week 6 所有 proof / cross-validation checks。

In [ ]:
checks = []

def add_check(name: str, passed: bool, detail: str = ''):
    checks.append({'check': name, 'passed': bool(passed), 'detail': detail})

add_check('schema_has_unique_row_id', df['row_id'].is_unique, f'n={len(df)}')
add_check('binary_labels_and_both_classes_exist', set(df['violation_label'].unique()).issubset({0, 1}) and df['violation_label'].nunique() == 2, str(df['violation_label'].value_counts().to_dict()))
add_check('label_severity_consistency', bool(((df['violation_label'] == 0) == (df['severity_score'].fillna(0) == 0)).all()), '')
add_check('pi_severity_dominates_component_margins', bool((df['target_pi_severity'] >= df['target_component_max_margin'] - 1e-12).all()), '')
add_check('no_leakage_feature_set', not set(feature_cols).intersection(leakage_cols), f'features={len(feature_cols)}')
add_check('no_suspicious_nonbase_feature_names', len(violating_feature_names) == 0, str(violating_feature_names))
add_check('disjoint_train_val_test', set(train_idx).isdisjoint(val_idx) and set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx), '')
add_check('preprocessed_features_are_finite', np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all(), '')
add_check('model_output_shapes_pass', shape_proof['passed'].all(), '')
add_check('finite_loss_and_gradients', grad_proof['passed'].all(), '')
add_check('test_probabilities_are_finite_and_bounded', np.isfinite(test_pi).all() and bool(((test_pi >= 0) & (test_pi <= 1)).all()), '')
add_check('manual_confusion_matrix_sums_to_test_n', manual_metrics['TP'] + manual_metrics['FP'] + manual_metrics['TN'] + manual_metrics['FN'] == len(test_idx), json.dumps(manual_metrics))
add_check('threshold_selected_from_validation', True, f'PI threshold={pi_threshold:.6f}')
add_check('scenario_group_cv_disjoint', bool((scenario_cv['group_disjoint'] == True).all()), '')
add_check('contingency_group_cv_disjoint', bool((contingency_cv['group_disjoint'] == True).all()), '')
add_check('fixed_seed_reproducibility', repro_max_abs_diff < 1e-6, f'max_diff={repro_max_abs_diff:.3e}')

validation_summary = pd.DataFrame(checks)
validation_summary.to_csv(OUT_DIR / 'week6_validation_summary.csv', index=False)

assert validation_summary['passed'].all(), validation_summary[~validation_summary['passed']]
validation_summary

## 19. Result summary JSON

最后保存一个 compact summary，方便后续 Week 7 / Week 8 引用。

In [ ]:
result_summary = {
    'n_samples': int(len(df)),
    'n_safe': int((df['violation_label'] == 0).sum()),
    'n_unsafe': int((df['violation_label'] == 1).sum()),
    'n_input_features_before_one_hot': int(len(feature_cols)),
    'encoded_feature_dim': int(input_dim),
    'target_recall': TARGET_RECALL,
    'selected_pi_threshold': pi_threshold,
    'validation_checks_passed': int(validation_summary['passed'].sum()),
    'validation_checks_total': int(len(validation_summary)),
    'best_holdout_model_by_fnr_then_saved_calls': holdout_results.sort_values(['fnr', 'pf_calls_saved_fraction'], ascending=[True, False]).iloc[0]['model'],
}

with open(OUT_DIR / 'week6_result_summary.json', 'w', encoding='utf-8') as f:
    json.dump(result_summary, f, indent=2)

result_summary

## 20. Student exercises

1. 修改 \(VUF^{max}\) 从 2% 到 1.5%，观察 component target 和模型结果是否变化。  
2. 关闭 `prob_sev` loss，比较 predicted probability 与 predicted severity 是否仍然单调。  
3. 把 `element_index` 改成 categorical feature，比较 contingency-group CV。  
4. 增加一个 cost-sensitive threshold：要求 FNR = 0，再最大化 PF calls saved。  
5. 把本 notebook 的 PI-MLP 替换为 Week 7 的 graph encoder。

## 21. Instructor notes

本 notebook 的课堂使用建议：

1. 先让学生运行到 no-leakage proof，确认哪些列不能作为输入；
2. 再运行普通 `BaselineMLP`，让学生观察它只能给出 violation probability；
3. 最后运行 `PI-MLP`，对比 probability、severity 和 component-margin heads；
4. 重点讨论 validation-threshold 选择，而不是追求单一最高 accuracy；
5. 把 `week6_loss_ablation_summary.csv` 和 `week6_group_cv_summary.csv` 作为论文实验表格雏形。

Week 7 会把这里的 tabular encoder 替换为 phase-aware graph encoder，但 high-recall screening、component targets 和 validation suite 可以继续沿用。